## StreetForward Training Demo (Golden Baseline Batch)

这个 notebook 演示如何：

- 使用配置 `configs/streetforward/multi_scene.yaml` 构建 `StreetForwardTrainer`
- 使用 `tools/record_streetforward_golden_baseline.py` 同款的 batch cache（来自 `utils.streetforward_baseline`）加载/生成 batch
- 将 batch 转换成 `StreetForwardTrainer.train_iter()` 需要的格式并运行一次训练迭代

> 说明：如果本地没有数据集（NuScenes）或没有 batch cache，本 notebook 仍会展示完整流程；生成 cache 需要真实数据集路径可用。


In [3]:
from __future__ import annotations

# 自动加载更改的魔法指令（在开发过程中自动重新加载模块）
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
import os
import torch

# 添加项目路径以导入模块
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.insert(0, project_root)

from utils.streetforward_baseline import (
    load_config,
    set_deterministic_seed,
    load_batch_cache,
    harvest_batch_cache,
    batch_plan_from_dataset,
    build_dataset,
    convert_batch_to_streetforward_format,
)
from models.streetforward.trainer import StreetForwardTrainer

CONFIG_PATH = f"{project_root}/configs/streetforward/multi_scene.yaml"
SEED = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("device:", DEVICE)
set_deterministic_seed(SEED)
cfg = load_config(CONFIG_PATH)
# keep absolute path for traceability (same as recorder)
cfg.config_path = os.path.abspath(CONFIG_PATH)

cfg

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
device: cuda


{'data': {'data_root': '/root/autodl-tmp/nuScenes/', 'dataset': 'nuscenes', 'start_timestep': 0, 'end_timestep': -1, 'preload_device': 'cpu', 'train_scene_ids': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9], 'eval_scene_ids': [10, 11, 12, 13], 'pixel_source': {'type': 'datasets.nuscenes.nuscenes_sourceloader.NuScenesPixelSource', 'cameras': [0, 1, 2], 'downscale_when_loading': [3, 3, 3], 'downscale': 1, 'undistort': False, 'test_image_stride': 0, 'max_test_images': 4, 'load_sky_mask': True, 'load_dynamic_mask': True, 'load_depth_maps': True, 'load_objects': True, 'load_smpl': False, 'sampler': {'buffer_downscale': 8, 'buffer_ratio': 0.5, 'start_enhance_weight': 3}}, 'lidar_source': {'type': 'datasets.nuscenes.nuscenes_sourceloader.NuScenesLiDARSource', 'load_lidar': True, 'lidar_downsample_factor': 4, 'lidar_percentile': 0.02}}, 'dataset': {'num_source_keyframes': 3, 'num_target_keyframes': 6, 'segment_overlap_ratio': 0.2, 'min_keyframes_per_scene': 10, 'min_keyframes_per_segment': 6, 'preload_scene_

In [5]:
# Batch cache path:
# - If you already generated one via tools/record_streetforward_golden_baseline.py,
#   point this to the same .pt.
# - Otherwise, set AUTO_HARVEST=True to generate it (requires dataset available).

BATCH_CACHE_PATH = Path("/root/drivestudio-coding/docs/trainers/golden/batch_cache_seed42_steps8_cuda.pt")
AUTO_HARVEST = False

if BATCH_CACHE_PATH.exists():
    meta, batches = load_batch_cache(BATCH_CACHE_PATH)
    print(f"Loaded batch cache: {BATCH_CACHE_PATH} (num_batches={len(batches)})")
    print("meta keys:", sorted(meta.keys()))
else:
    print(f"Batch cache not found: {BATCH_CACHE_PATH}")
    if not AUTO_HARVEST:
        raise FileNotFoundError(
            "No batch cache found. Either set BATCH_CACHE_PATH to an existing .pt created by "
            "tools/record_streetforward_golden_baseline.py --batch-cache ... --harvest-batch-cache, "
            "or set AUTO_HARVEST=True to generate one (requires dataset access)."
        )

    dataset = build_dataset(cfg, DEVICE)
    plan = batch_plan_from_dataset(dataset, max_scenes=2, segments_per_scene=2, batches_per_segment=2)
    print("Harvest plan:", plan)
    meta = harvest_batch_cache(
        cfg=cfg,
        device=DEVICE,
        seed=SEED,
        plan=plan,
        output_path=BATCH_CACHE_PATH,
        include_test=False,
    )
    meta, batches = load_batch_cache(BATCH_CACHE_PATH)
    print(f"Harvested & loaded batch cache: {BATCH_CACHE_PATH} (num_batches={len(batches)})")

batch0 = batches[0]
print("batch0 keys:", sorted(batch0.keys()))
print("scene_id, segment_id:", batch0.get("scene_id"), batch0.get("segment_id"))
batch0

Loaded batch cache: /root/drivestudio-coding/docs/trainers/golden/batch_cache_seed42_steps8_cuda.pt (num_batches=8)
meta keys: ['config_path', 'include_test', 'plan', 'scene_segment_sequence', 'seed']
batch0 keys: ['dynamic_info', 'keyframe_info', 'pointcloud', 'scene_folder_name', 'scene_id', 'segment_id', 'source', 'target']
scene_id, segment_id: tensor([7]) 0


{'scene_id': tensor([7]),
 'scene_folder_name': '007',
 'segment_id': 0,
 'keyframe_info': {'segment_keyframes': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
  'source_keyframes': [9],
  'target_keyframes': [9, 6, 0, 8, 7, 1]},
 'source': {'image': tensor([[[[0.2902, 0.3333, 0.2745],
            [0.2745, 0.3098, 0.2510],
            [0.2392, 0.2667, 0.2118],
            ...,
            [0.2078, 0.2392, 0.2471],
            [0.1843, 0.2157, 0.2275],
            [0.1608, 0.1922, 0.2039]],
  
           [[0.2157, 0.2510, 0.2353],
            [0.2314, 0.2627, 0.2353],
            [0.2431, 0.2706, 0.2275],
            ...,
            [0.2392, 0.2745, 0.2667],
            [0.2275, 0.2627, 0.2549],
            [0.2039, 0.2353, 0.2314]],
  
           [[0.2157, 0.2431, 0.2667],
            [0.2863, 0.3098, 0.3098],
            [0.3059, 0.3333, 0.2941],
            ...,
            [0.2510, 0.2863, 0.2667],
            [0.2510, 0.2863, 0.2667],
            [0.2353, 0.2706, 0.2549]],
  
           ...,

In [7]:
# Convert MultiSceneDataset batch -> StreetForward batch format
street_batch0 = convert_batch_to_streetforward_format(batch0, DEVICE)

print("street_batch0 keys:", sorted(street_batch0.keys()))
print("num_targets:", len(street_batch0.get("targets", [])))
print("source_frame_idx:", street_batch0.get("source_frame_idx"))

# quick sanity checks
assert "pointcloud" in street_batch0
assert "targets" in street_batch0
assert isinstance(street_batch0["targets"], list)
street_batch0["targets"][0]

street_batch0 keys: ['dynamic_info', 'gt_images', 'pointcloud', 'scene_id', 'segment_id', 'source_frame_idx', 'source_views', 'src_images', 'target_views', 'targets', 'test_images', 'test_views']
num_targets: 18
source_frame_idx: 34


{'frame_idx': 36,
 'view': <utils.streetforward_baseline.View at 0x7f09d438e7c0>,
 'gt_image': tensor([[[0.3059, 0.3020, 0.2824],
          [0.2471, 0.2392, 0.2392],
          [0.2157, 0.2118, 0.2196],
          ...,
          [0.2000, 0.2118, 0.2078],
          [0.2000, 0.2118, 0.2078],
          [0.2157, 0.2275, 0.2235]],
 
         [[0.3490, 0.3451, 0.3373],
          [0.2980, 0.2902, 0.2980],
          [0.2667, 0.2627, 0.2824],
          ...,
          [0.1961, 0.2118, 0.2078],
          [0.2078, 0.2235, 0.2196],
          [0.2275, 0.2431, 0.2392]],
 
         [[0.3333, 0.3294, 0.3255],
          [0.3098, 0.3098, 0.3176],
          [0.2980, 0.2941, 0.3176],
          ...,
          [0.1961, 0.2157, 0.2118],
          [0.1961, 0.2157, 0.2078],
          [0.1961, 0.2157, 0.2078]],
 
         ...,
 
         [[0.2667, 0.2824, 0.2588],
          [0.3059, 0.3216, 0.2980],
          [0.3020, 0.3176, 0.2941],
          ...,
          [0.4824, 0.4980, 0.5098],
          [0.4863, 0.5020, 0.

In [8]:
# Build trainer from config and run a single training iteration
trainer = StreetForwardTrainer(config=cfg, device=DEVICE)
trainer.train()

result = trainer.train_iter(
    batch=street_batch0,
    apply_update=True,
    update_state=True,
    evaluate_test=False,
)

print("total_loss:", float(result["total_loss"].detach().item()) if torch.is_tensor(result["total_loss"]) else result["total_loss"])
print("node_state_bg num_points:", int(result["node_state"].means.shape[0]))
print("node_state_rigid present:", result.get("node_state_rigid") is not None)
print("node_state_distant present:", result.get("node_state_distant") is not None)

# Optional: inspect cached summaries on trainer (recording script uses these)
print("has _last_offsets_bg:", hasattr(trainer, "_last_offsets_bg"))
if getattr(trainer, "_last_offsets_bg", None) is not None:
    print("offset_bg keys:", list(trainer._last_offsets_bg.keys()))

result

/root/autodl-tmp/conda/envs/drivestudio-new/lib/python3.9/site-packages/torch/utils/cpp_extension.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import packaging  # type: ignore[attr-defined]


total_loss: 0.09937817603349686
node_state_bg num_points: 500000
node_state_rigid present: True
node_state_distant present: False
has _last_offsets_bg: True
offset_bg keys: ['offset_pos', 'offset_scales', 'offset_quat', 'offset_opacity', 'offset_sh']


{'total_loss': tensor(0.0994, device='cuda:0'),
 'node_state': NodeStateBackground(means=tensor([[-13.4523,  -3.5151,  20.2973],
         [ 14.7432,  -1.5166,  16.4739],
         [  5.9930,  -4.4572,  32.5827],
         ...,
         [ -0.2030,   0.1682,  29.1897],
         [  7.2099,  -3.2312,  15.9878],
         [ -4.9137,   0.6600,  18.5783]], device='cuda:0'), scales_log=tensor([[-1.8145, -1.8145, -1.8145],
         [-1.6357, -1.6357, -1.6357],
         [-1.9762, -1.9762, -1.9762],
         ...,
         [-2.3122, -2.3122, -2.3122],
         [-1.7119, -1.7119, -1.7119],
         [-2.2465, -2.2465, -2.2465]], device='cuda:0'), quats=tensor([[-0.3616, -0.5528,  0.2855,  0.6944],
         [ 0.1003,  0.9679,  0.2305, -0.0021],
         [ 0.5700, -0.7690, -0.1011,  0.2711],
         ...,
         [-0.1965,  0.3203,  0.5700,  0.7307],
         [-0.0927, -0.0127, -0.6200,  0.7790],
         [-0.0739,  0.4527,  0.3845,  0.8011]], device='cuda:0'), opacity_logit=tensor([[-2.1972],
         

In [9]:
# (Optional) Run a few steps over cached batches to show trainer state evolves
NUM_STEPS = min(3, len(batches))
losses = []

for i in range(NUM_STEPS):
    street_batch = convert_batch_to_streetforward_format(batches[i], DEVICE)
    out = trainer.train_iter(street_batch, apply_update=True, update_state=True)
    loss_val = float(out["total_loss"].detach().item()) if torch.is_tensor(out["total_loss"]) else float(out["total_loss"])
    losses.append(loss_val)
    sid = int(street_batch["scene_id"].item())
    seg = int(street_batch["segment_id"].item())
    print(f"step={i} scene={sid} segment={seg} loss={loss_val:.6f}")

losses

step=0 scene=7 segment=0 loss=0.099369
step=1 scene=7 segment=1 loss=0.126491
step=2 scene=7 segment=2 loss=0.140295


[0.09936946630477905, 0.12649133801460266, 0.14029479026794434]